# ronin-code-1.5b — FREE training on Colab (T4, $0)

Trains Ronin's QLoRA adapter on the free Colab T4. **Before running:** `Runtime → Change runtime type → T4 GPU`.

Honesty rails: the frozen test split is hash-verified before training; the eval never invents a score;
the ship gate is pre-committed — **valid_tool_json 4/4 AND >60/91** on the frozen 91-case eval, or archive.

Free-tier reality (plan around it): sessions time out (~hours) and can disconnect at any moment —
the trainer checkpoints every 100 steps, so add `--resume` to Cell 3 and re-run after a disconnect.
T4 availability is not guaranteed. Fallback: **Kaggle** (30 GPU-hrs/week free) — same cells verbatim.


In [ ]:
# Cell 1 — clone + pinned deps (a few minutes)
!git clone https://github.com/rohithkandula19/Ronin.git /content/Ronin 2>/dev/null || (cd /content/Ronin && git pull)
%cd /content/Ronin
!pip install -q -r training/cuda/requirements.txt
!pip install -q -e packages/dialect -e training
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > T4!')


In [ ]:
# Cell 2 — deterministic corpus (post D1/D2 fix) + frozen-split verification
!python -m ronin_training.dataset_builder --volumes docs/volumes --out training/data/generated
!python -m ronin_training.synthetic_corpus --target 5000 --seed 42 --out training/data/synthetic
!mkdir -p training/data/merged
!for s in train valid; do cat training/data/generated/$s.jsonl training/data/synthetic/$s.jsonl > training/data/merged/$s.jsonl; done
!cp training/data/synthetic/test.jsonl training/data/merged/test.jsonl
!cp training/data/synthetic/test.jsonl.sha256 training/data/merged/test.jsonl.sha256
# preflight: config + FROZEN SPLIT hash — refuses to train on a tampered split
!python training/cuda/train_cuda.py --check --qlora --fp16 --data training/data/merged


In [ ]:
# Cell 3 — QLoRA train on the T4 (fp16 + 4-bit nf4, seed 42, ckpt every 100 steps)
# After a disconnect: re-run Cells 1-2, then add --resume here.
!python training/cuda/train_cuda.py --qlora --fp16 --save-steps 100 \
    --data training/data/merged --out training/adapters/ronin-code-1.5b-v5


In [ ]:
# Cell 4 — make the artifact survive the session: Google Drive + the free-HF upload command
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ronin-adapters
!cp -r training/adapters/ronin-code-1.5b-v5/adapters /content/drive/MyDrive/ronin-adapters/ronin-code-1.5b-v5
print('Adapter saved to Drive: MyDrive/ronin-adapters/ronin-code-1.5b-v5')
print()
print('Publish with a FREE Hugging Face account (only after the eval clears the gate):')
print('  pip install -U huggingface_hub && huggingface-cli login')
print('  hf upload <your-hf-user>/ronin-code-1.5b training/adapters/ronin-code-1.5b-v5/adapters --repo-type model')


In [ ]:
# Cell 5 — the honest eval: frozen 91-case set, baseline delta, provenance-stamped report
# Scores land in training/reports/ with model+adapter+commit SHA. NO score is ever invented.
!python -m ronin_training.eval_runner \
    --provider hf --model Qwen/Qwen2.5-Coder-1.5B-Instruct \
    --adapter training/adapters/ronin-code-1.5b-v5/adapters --baseline
print('Gate: ship ONLY if valid_tool_json 4/4 AND >60/91. Miss it -> archive, honestly.')
